### P2.3 — Deep Learning Foundations | Python to Gen AI Course
### P2.3.8 — RNN & LSTM — Sequence Learning


---
## 🔁 Where We Are — The Big Picture

So far in this course we've built two types of neural networks:

```
ANN  →  Tabular data     (rows and columns — order does NOT matter)
CNN  →  Image data       (pixels arranged in a grid — spatial structure matters)
```

ANN and CNN have one important thing in common — they process each input **independently**, with no memory of what came before.

That works perfectly for house prices and images.
But for sequences — text, audio, time series — **order is everything.**

```
"The movie was not bad at all"   →  Positive sentiment
"The movie was bad, not good"    →  Negative sentiment

Same words. Different order. Completely different meaning.
```

An ANN sees only a **bag of words** — it cannot tell these two sentences apart.
An RNN reads the sequence **in order** — and remembers what came before.


---
# RNN (Recurrent Neural Network)

## 🧠 The Core Idea — Memory

A regular ANN neuron processes one input and immediately forgets it:
```
Input  →  Neuron  →  Output
          (forgets immediately)
```

An RNN neuron does something different — it has a **hidden state** that acts as memory:
```
Input  →  Neuron  →  Output
            ↑  ↓
         Hidden State
         (memory passed to the next step)
```

At every time step, the RNN receives **two things**:
1. The **current input** (the new word / data point)
2. The **hidden state** from the previous step (memory of everything before)

It combines both, produces an output, and passes an updated hidden state to the next step.

---

### 📖 Example — Reading a Sentence Step by Step

```
Time Step 1:  Input = 'The'  + no memory yet   →  Output + Hidden State h1
Time Step 2:  Input = 'cat'  + h1              →  Output + Hidden State h2
Time Step 3:  Input = 'sat'  + h2              →  Output + Hidden State h3
Time Step 4:  Input = 'on'   + h3              →  Output + Hidden State h4
Time Step 5:  Input = 'the'  + h4              →  Output + Hidden State h5
Time Step 6:  Input = 'mat'  + h5              →  FINAL OUTPUT
```

By the final step, the hidden state has seen **every single word** in the sentence.
That final hidden state is used to make the prediction.

> 💡 Think of the hidden state like a **running summary** —
> each step it updates itself based on the new word + what it already remembered.

<img src="images/rnn_unrolled.png" width="850"/>

---
## ⚠️ The Vanishing Gradient Problem

Plain RNNs have a serious limitation — **short memory.**

Recall from earlier in the course: gradients flow **backwards** through the network during backpropagation. In an RNN, gradients also flow backwards through **time steps.**

```
Step 100  ←  Step 99  ←  Step 98  ←  ...  ←  Step 1

At each step, the gradient gets multiplied by a small number.
After many steps:  0.9 × 0.9 × 0.9 × 0.9 × ...  ≈  almost zero
```

The gradient **vanishes** before it reaches early time steps.
Early steps receive almost no learning signal — so the model simply **forgets the distant past.**

### 🔴 Real Consequence

```
Sentence:  'The cat, which was sitting on the old dusty mat in the corner, sat.'

Plain RNN remembers :  '...in the corner, sat'   (recent words ✅)
Plain RNN forgets   :  'The cat...'              (distant subject ❌)

→ It cannot connect 'cat' with 'sat' — loses the meaning of the sentence
```

This is called a **long-range dependency** — the RNN cannot handle it.

<img src="images/vanishing_gradient.png" width="750"/>
---

### 📌 RNN Summary

| Concept | Key Point |
|---|---|
| Why RNN exists | ANN has no memory — order is invisible. RNN reads sequences in order |
| Hidden State | Memory passed step to step — carries context forward |
| Vanishing Gradient | Gradients shrink to near-zero over long sequences — plain RNN forgets the distant past |
| Limitation | Cannot handle long-range dependencies — fixed by LSTM |

---
# LSTM (Long Short-Term Memory)

## 🔒 LSTM — The Fix for Short Memory

**LSTM (Long Short-Term Memory)** was designed specifically to solve the vanishing gradient problem.

The key innovation: an LSTM cell has **two memory tracks**, not one:

```
Hidden State  →  short-term memory  (same as plain RNN)
Cell State    →  long-term memory   (NEW — runs straight through all steps)
```

The **cell state** is like a conveyor belt — information travels along it across many steps without being multiplied away. This is what prevents the vanishing gradient.

---

## 🚪 The Three Gates — How LSTM Controls Memory

LSTM controls what to remember and what to forget using three **gates**.
Each gate is a small neural network that learns what to do during training.

```
────────────────────────────────────────────────────────────────────────
 Gate            What it decides                         Example
────────────────────────────────────────────────────────────────────────
 Forget Gate     What to erase from long-term memory     Forget old subject
                                                          when a new sentence starts

 Input Gate      What new information to write           Remember the new subject
                 into long-term memory                   being introduced

 Output Gate     What to pass forward as the            What's relevant
                 hidden state (short-term output)        for the current prediction
────────────────────────────────────────────────────────────────────────
```

> 💡 You do **not** need to implement these gates manually.
> `keras.layers.LSTM()` handles all of this internally.
> Understanding gates helps you understand **why** LSTM works better than plain RNN.

<img src="images/lstm_gates.png" width="800"/>

---
## 🆚 RNN vs LSTM — Side-by-Side Comparison

```
──────────────────────────────────────────────────────────────────────
                     Plain RNN              LSTM
──────────────────────────────────────────────────────────────────────
 Memory type         Short-term only        Short-term + Long-term
 Long sequences      Struggles              Handles well
 Vanishing gradient  Yes — big problem      Mostly solved
 Speed               Faster                 Slightly slower
 Parameters          Fewer                  More (4x gating logic)
 Used in practice    Rarely used alone      Standard choice
 Best for            Very short sequences   Most sequence tasks
──────────────────────────────────────────────────────────────────────
```

> ✅ **Rule of thumb: always start with LSTM.**
> Plain RNN is mostly used for teaching concepts.
> LSTM is what gets used in real projects.

---
# Project 1: Sentiment Analysis (IMDB Reviews)

**Task:** Classify a movie review as Positive or Negative.

**Dataset:** IMDB — 50,000 movie reviews, pre-labeled (built into Keras).

```
Input  →  a sequence of words (up to 200 words per review)
Output →  1 neuron, sigmoid activation  →  Positive (1) or Negative (0)
```

**Architecture:**
```
Embedding Layer  →  converts each word into a list of numbers that captures its meaning (learned during training)
LSTM Layer       →  reads the sequence, maintains memory across all 200 words
Dropout Layer    →  prevents overfitting
Dense Output     →  sigmoid → probability of being positive
```

### Phase 1 — Load Data

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# IMDB is built into Keras — words are already converted to integers
# num_words=10000 → keep only the 10,000 most common words (ignore rare words)
max_words  = 10000
max_length = 200   # use only the first 200 words of each review

(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(
    num_words=max_words
)

print(f"Train reviews : {len(X_train)}")
print(f"Test  reviews : {len(X_test)}")
print(f"Sample label  : {y_train[0]}   (1 = Positive, 0 = Negative)")
print(f"Sample review : {X_train[0][:10]}...  (word indices, not actual words)")

### Phase 2 — Pad Sequences

Every review has a different length. LSTM needs all inputs to be the **same length**.

- Short reviews → pad with zeros at the start (pre-padding)
- Long reviews  → trim to `max_length`

In [ ]:
X_train = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=max_length)
X_test  = keras.preprocessing.sequence.pad_sequences(X_test,  maxlen=max_length)

print(f"Shape after padding: {X_train.shape}")
print(f"Each review is now exactly {max_length} word indices long")

### Phase 3 — Build & Train the Model

**What each layer does:**
- `Embedding` — maps each word index to a 64-dimensional vector. Similar words learn similar vectors. This is how the model understands meaning.
- `LSTM(64)` — reads all 200 words in sequence, maintaining memory. Returns only the final step's output.
- `Dropout(0.3)` — randomly turns off 30% of neurons during training to prevent overfitting.
- `Dense(1, sigmoid)` — final decision: positive or negative.

In [ ]:
model = keras.Sequential([
    layers.Embedding(max_words, 64, input_length=max_length),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=128,
    verbose=1
)

### Phase 4 — Evaluate

In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss,  test_acc  = model.evaluate(X_test,  y_test,  verbose=0)

train_acc = round(train_acc * 100, 2)
test_acc  = round(test_acc  * 100, 2)

print("=== EVALUATE ===")
print(f"  Train Accuracy : {train_acc}%")
print(f"  Test  Accuracy : {test_acc}%")

gap = train_acc - test_acc
if gap < 5:
    print("  ✅ Good Fit — train and test scores are close")
elif gap > 15:
    print("  ⚠️  Overfitting — try adding more Dropout or reducing LSTM units")
else:
    print("  ✅ Acceptable — small gap, generalises reasonably well")

### Phase 5 — Inference (Predict on Your Own Reviews)

In [ ]:
import re
import numpy as np

# Get the IMDB word-to-index dictionary
word_index = keras.datasets.imdb.get_word_index()

def predict_sentiment(review_text):
    """
    Takes a raw text review and returns a sentiment prediction.
    Steps: clean text → convert to word indices → pad → predict
    """
    words   = re.sub(r'[^a-z ]', '', review_text.lower()).split()
    indices = [word_index.get(w, 2) for w in words]   # 2 = unknown word token
    indices = indices[:max_length]

    padded = keras.preprocessing.sequence.pad_sequences([indices], maxlen=max_length)
    prob   = model.predict(padded, verbose=0)[0][0]

    label = "POSITIVE" if prob > 0.5 else "NEGATIVE"
    conf  = round(prob * 100 if prob > 0.5 else (1 - prob) * 100, 1)
    return label, conf


print("=== INFERENCE ===")
reviews = [
    "This movie was absolutely fantastic. The acting was brilliant.",
    "Terrible film. Complete waste of time. Boring and predictable.",
    "Not bad. Some good moments but overall quite average.",
]
for review in reviews:
    label, conf = predict_sentiment(review)
    print(f"  Review  : '{review[:60]}...'")
    print(f"  Result  : {label}  ({conf}% confident)")
    print()

---
# Stock Price Prediction (AAPL)

**Task:** Predict the next day's Apple stock price using the last 60 days.

**Why LSTM for stock prices?**
Stock prices are a **time series** — each value depends on what came before.
LSTM's memory makes it well-suited for learning patterns in sequential price data.

```
Input  →  60 days of closing prices (a sequence)
Output →  predicted price for day 61
Loss   →  MSE (Mean Squared Error) — this is regression, not classification
```

**Note:** This is a **learning exercise** — stock prediction is genuinely hard.
A good MAE here means the model learned the general trend, not the future.

### Step 1 — Install & Import

In [ ]:
# Run this once (on Colab or first time locally)
! pip install yfinance scikit-learn
print("Libraries ready ✅")

In [ ]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers

print("All imports done ✅")

### Step 2 — Download & Visualize Data

In [ ]:
# Download Apple (AAPL) stock — 5 years of daily closing prices
stock = yf.download('AAPL', start='2019-01-01', end='2024-01-01')
data  = stock['Close'].values.reshape(-1, 1)

print(f"Stock      : Apple (AAPL)")
print(f"Total days : {len(data)}")
print(f"Date range : 2019 to 2024")

plt.figure(figsize=(12, 4))
plt.plot(stock['Close'])
plt.title("Apple Stock Price — 5 Years")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.grid(True)
plt.show()

### Step 3 — Normalize + Create Sequences

**Why normalize?**
LSTM trains much better on small numbers (0 to 1) than on large ones ($100–$200).
We use MinMaxScaler — same idea as normalizing pixel values in CNN.

**Why sequences?**
LSTM needs sequences as input. We create samples like this:
```
Sample 1:  days 1–60   → predict day 61
Sample 2:  days 2–61   → predict day 62
Sample 3:  days 3–62   → predict day 63
... and so on
```

In [ ]:
# Normalize to 0-1
scaler      = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# Create sequences
SEQ_LENGTH = 60   # look back 60 days to predict day 61

X, y = [], []
for i in range(SEQ_LENGTH, len(data_scaled)):
    X.append(data_scaled[i-SEQ_LENGTH:i, 0])   # 60 days as input
    y.append(data_scaled[i, 0])                 # day 61 as target

X = np.array(X)
y = np.array(y)

# LSTM input shape must be: (samples, time steps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

# 80/20 train-test split
split   = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Sequence length : {SEQ_LENGTH} days")
print(f"X_train shape   : {X_train.shape}   → (samples, time_steps, features)")
print(f"X_test shape    : {X_test.shape}")

### Step 4 — Build the Stacked LSTM Model

We use **two LSTM layers** (stacked). The first layer learns basic patterns, the second learns deeper patterns built on top.

- `return_sequences=True` on the first LSTM → passes the full sequence to the second LSTM
- `return_sequences=False` on the second LSTM → passes only the final step output forward
- No activation on the output layer → this is regression (predicting a number, not a class)

In [ ]:
model_stock = keras.Sequential([
    layers.LSTM(50, return_sequences=True, input_shape=(SEQ_LENGTH, 1)),
    layers.LSTM(50, return_sequences=False),
    layers.Dense(25, activation='relu'),
    layers.Dense(1)   # no activation — regression output
])

model_stock.compile(
    optimizer='adam',
    loss='mean_squared_error'   # MSE for regression (not binary_crossentropy)
)

model_stock.summary()

### Step 5 — Train

In [ ]:
# Colab GPU  → ~2–3 minutes
# Local CPU  → ~10–15 minutes

history_stock = model_stock.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    verbose=1
)

plt.figure(figsize=(10, 4))
plt.plot(history_stock.history['loss'],     label='Train Loss')
plt.plot(history_stock.history['val_loss'], label='Val Loss')
plt.title("Training Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True)
plt.show()

### Step 6 — Evaluate

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

predictions_scaled = model_stock.predict(X_test, verbose=0)

# Convert predictions back from normalized (0–1) to real dollar prices
predictions = scaler.inverse_transform(predictions_scaled)
actual      = scaler.inverse_transform(y_test.reshape(-1, 1))

mse = mean_squared_error(actual, predictions)
mae = mean_absolute_error(actual, predictions)

print("=== EVALUATE ===")
print(f"  MSE : {round(mse, 2)}  (mean squared error in dollars²)")
print(f"  MAE : ${round(mae, 2)}  (on average, predicted price is off by this much)")

### Step 7 — Visualize Predictions vs Actual

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(actual,      label='Actual Price',    color='blue')
plt.plot(predictions, label='Predicted Price', color='orange', linestyle='--')
plt.title("Apple Stock — Actual vs Predicted Price")
plt.xlabel("Days (Test Set)")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True)
plt.show()

# A good result: the orange line closely follows the blue line

### Step 8 — Inference (Predict the Next Day's Price)

In [ ]:
# Use the last 60 known days to predict tomorrow's price
last_60_days        = data_scaled[-60:].reshape(1, 60, 1)
next_day_scaled     = model_stock.predict(last_60_days, verbose=0)
next_day_price      = scaler.inverse_transform(next_day_scaled)

print("=== INFERENCE ===")
print(f"  Last known price : ${round(float(data[-1]), 2)}")
print(f"  Predicted next   : ${round(float(next_day_price[0][0]), 2)}")

---
# Project 3: Speech-to-Text with Whisper

**What is Whisper?**
Whisper is a pretrained model from OpenAI that converts **spoken audio → text**.
It was trained on 680,000 hours of audio — we just load it and use it directly.

**Why is this here (in the RNN/LSTM notebook)?**
Audio is a **sequence** — sound waves over time. Under the hood, models like Whisper
use sequence-processing architectures (Transformers, which evolved from RNN/LSTM ideas).
This shows a real-world application of sequence learning.

```
Audio file  →  Whisper (pretrained)  →  Transcribed text + detected language
```

No training needed — this is **inference only** on a pretrained model.
Whisper model sizes:
```
tiny  →  fastest, least accurate
base  →  good balance for learning  ← we use this
small / medium / large  →  more accurate, slower
```

### Step 1 — Install Whisper

In [ ]:
! pip install openai-whisper
! pip install sounddevice scipy
print("Whisper installed ✅")

### Step 2 — Load the Pretrained Model

In [ ]:
import whisper

# Load pretrained Whisper — downloads weights automatically on first run
whisper_model = whisper.load_model("base")

print("Whisper loaded ✅")
print("Pretrained on 680,000 hours of audio")
print("No training needed — use directly for inference")

### Step 3 — Option A: Transcribe from an Audio File

Supported formats: `.mp3`  `.wav`  `.m4a`  `.ogg`  `.flac`

In [ ]:
filename = input("Enter full path to your audio file: ")

# One line to transcribe — Whisper handles everything internally
result = whisper_model.transcribe(filename)

print("\n=== SPEECH TO TEXT RESULT ===")
print(f"Text     : {result['text']}")
print(f"Language : {result['language']}")

### Step 4 — Option B: Record from Your Microphone and Transcribe

This records 5 seconds from your mic, saves it as a `.wav` file, then runs Whisper on it.

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import numpy as np
import time

DURATION    = 5       # seconds to record — change as needed
SAMPLE_RATE = 16000   # Whisper expects 16000 Hz (16 kHz)

print("🎙️  Recording starts in 3 seconds...")
print("Speak clearly after the prompt!")
time.sleep(3)

print("🔴 Recording NOW...")
audio = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,         # mono — one microphone
    dtype='float32'
)
sd.wait()   # block until recording is complete
print("⏹️  Recording done ✅")

# Save to file
write("my_recording.wav", SAMPLE_RATE, audio)
print("Saved as my_recording.wav ✅")

# Transcribe with Whisper
result = whisper_model.transcribe("my_recording.wav")

print("\n=== SPEECH TO TEXT RESULT ===")
print(f"Text     : {result['text']}")
print(f"Language : {result['language']}")

---
# ✅ Full Summary — RNN & LSTM

| Topic | Key Takeaway |
|---|---|
| Why RNN | ANN ignores order. RNN reads sequences step by step |
| Hidden State | Memory that flows from step to step |
| Vanishing Gradient | Gradients shrink to zero over long sequences — plain RNN forgets the distant past |
| LSTM Fix | Cell state (long-term memory) + gating = gradients survive long sequences |
| Forget Gate | Decides what to erase from long-term memory |
| Input Gate | Decides what new info to store in long-term memory |
| Output Gate | Decides what to pass forward as the current hidden state |
| Sentiment Project | Embedding + LSTM → binary classification of text |
| Stock Project | Stacked LSTM → regression on time series data |
| Whisper Project | Pretrained model — audio (a sequence) → transcribed text |
| Practical rule | Always use LSTM over plain RNN in real projects |